In [6]:
import os
import pandas as pd
import numpy as np
import pickle

# Define relative paths based on the project structure
RAW_DATA_DIR = '/Users/rohitsuresh/Desktop/CGT_Carbon_farming/data/Dataset_INR'
PROCESSED_DATA_DIR = '/Users/rohitsuresh/Desktop/CGT_Carbon_farming/data/processed'

# Create the processed directory if it doesn't exist
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

# Define Big-M penalty for incompatible practices (-inf in your CSVs)
# This strongly penalizes the objective function if an incompatible pair is selected
BIG_M_PENALTY = -1e6

In [8]:
farmers_df = pd.read_csv(f'{RAW_DATA_DIR}/15_farmers.csv')

farmer_ids = farmers_df['Farmer_ID'].tolist()
farm_sizes = dict(zip(farmers_df['Farmer_ID'], farmers_df['Farm_Size_ha']))
farmer_budgets = dict(zip(farmers_df['Farmer_ID'], farmers_df['Budget_total_INR_per_season']))

print(f"Loaded {len(farmer_ids)} farmers.")

practices_df = pd.read_csv(f'{RAW_DATA_DIR}/revised_20_practice_interaction_matrices_v3.csv', encoding='cp1252')
practice_ids = practices_df['Practice'].tolist()


base_csp = dict(zip(practices_df['Practice'], practices_df['Net_CSP_base']))
base_oc = dict(zip(practices_df['Practice'], practices_df['Net_OC']))

print(f"Loaded {len(practice_ids)} practices.")

Loaded 15 farmers.
Loaded 20 practices.


In [10]:
def process_interaction_matrix(filename, practices_list, big_m_penalty):
    """
    Reads a square interaction matrix, replaces -inf, and flattens it 
    into a dictionary of upper-triangle pairwise combinations.
    """
    filepath = f'{RAW_DATA_DIR}/{filename}'
    df = pd.read_csv(filepath, index_col=0, encoding='cp1252')
    
    df = df.replace('-inf', big_m_penalty).replace([np.inf, -np.inf], big_m_penalty)
    df = df.fillna(0)
    
    interaction_dict = {}
    
    # Extract upper triangle since interactions are symmetric i < j
    for i, p1 in enumerate(practices_list):
        for j, p2 in enumerate(practices_list):
            if i < j: 
                # Ensure the keys exist in the dataframe to avoid KeyError
                if p1 in df.index and p2 in df.columns:
                    val = float(df.loc[p1, p2])
                    interaction_dict[(p1, p2)] = val
                else:
                    interaction_dict[(p1, p2)] = 0.0
                    
    return interaction_dict

# Process Alpha (Carbon), Beta (Cost), and Gamma (Yield)
alpha_carbon = process_interaction_matrix('alpha_carbon.csv', practice_ids, BIG_M_PENALTY)
beta_cost = process_interaction_matrix('Beta_cost.csv', practice_ids, BIG_M_PENALTY)
gamma_yield = process_interaction_matrix('gamma_yield.csv', practice_ids, BIG_M_PENALTY)

print(f"Processed interaction matrices. Each contains {len(alpha_carbon)} unique pairs.")

Processed interaction matrices. Each contains 190 unique pairs.


In [ ]:
# Consolidate all parameters into a single payload
processed_data = {
    'farmer_ids': farmer_ids,
    'farm_sizes': farm_sizes,
    'farmer_budgets': farmer_budgets,
    'practice_ids': practice_ids,
    'base_csp': base_csp,
    'base_oc': base_oc,
    'alpha_carbon': alpha_carbon,
    'beta_cost': beta_cost,
    'gamma_yield': gamma_yield,
    'constants': {
        'BIG_M': BIG_M_PENALTY,
        'CCP': 1350,
        'MRV_FIXED': 30000,
        'MRV_VARIABLE': 2000,
        'MRV_DELTA': 0.8,
        'T_FIXED': 10000,
        'T_VARIABLE': 500
    }
}

# Save to disk using pickle
export_path = f'{PROCESSED_DATA_DIR}/optimization_inputs_15_farmers.pkl'
with open(export_path, 'wb') as f:
    pickle.dump(processed_data, f)

print(f"Data successfully exported to {export_path}")

Data successfully exported to /Users/rohitsuresh/Desktop/CGT_Carbon_farming/data/processed/optimization_inputs_15_farmers.pkl
